# Where Should 40 Analyst Hours Go?

**Analytics Skills Challenge — Talay Kamali**

20 analysts × 2 hrs/week = 40 hrs/week of skilled analytical capacity. The brief asks where to invest it. This notebook treats that as a *matched-resource* problem, not a donation-ranking problem — analyst time goes where analytical capacity is the bottleneck, not where money is needed most.

**Sources** (both data.gov.au, official):
- **AIS 2020** — Annual Information Statement (financials, staff, volunteers)  
  https://data.gov.au/data/dataset/acnc-2020-annual-information-statement-data/resource/9eda5c10-bded-410e-8ee9-a327a8ff2560
- **ACNC Charity Register** — current mission, beneficiaries, geography  
  https://data.gov.au/data/dataset/acnc-register/resource/eb1e6be4-5b13-4feb-b28e-388bf7c26f93

**Output**: scored shortlist exported to CSV for the Tableau dashboard.

## 1. Setup & load

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
sns.set_style('whitegrid')

ais = pd.read_excel('datadotgov_ais20.xlsx')
reg = pd.read_excel('datadotgov_main.xlsx')

print(f'AIS 2020:       {len(ais):>7,} rows × {ais.shape[1]} cols')
print(f'ACNC Register:  {len(reg):>7,} rows × {reg.shape[1]} cols')

AIS 2020:        51,392 rows × 93 cols
ACNC Register:   65,629 rows × 69 cols


## 2. Join on ABN

ABN is the unique charity identifier in both datasets. The Register has more rows because it includes charities registered after the 2020 filing year — that's expected and fine; the inner join keeps charities with both financials and current mission data.

In [2]:
# Normalize join keys (float vs int casting issues)
reg_clean = reg.dropna(subset=['ABN']).copy()
ais['abn_key'] = ais['abn'].astype('int64').astype(str)
reg_clean['abn_key'] = reg_clean['ABN'].astype('int64').astype(str)

df = ais.merge(reg_clean, on='abn_key', how='inner')
print(f'Joined: {len(df):,} charities with both financial and mission data')
print(f'Retention: {len(df)/len(ais):.1%} of AIS records')

Joined: 44,843 charities with both financial and mission data
Retention: 87.3% of AIS records


## 3. Sector snapshot — what 45k charities look like

Before scoring, the sector itself. Three views that set up the narrative: size distribution, revenue concentration, and where volunteers actually live.

In [3]:
# Size distribution
size_dist = df['charity size'].value_counts()
print('Charity size:')
print(size_dist)
print(f'\nSmall charities are {size_dist.get("Small",0)/len(df):.0%} of the sector.')

Charity size:
charity size
Small     29370
Large      8104
Medium     7367
Name: count, dtype: int64

Small charities are 65% of the sector.


In [4]:
# Revenue concentration — Pareto check
rev = df['total revenue'].fillna(0).sort_values(ascending=False)
cum_pct = rev.cumsum() / rev.sum()
n_for_80pct = (cum_pct <= 0.80).sum() + 1
print(f'{n_for_80pct:,} charities ({n_for_80pct/len(df):.1%}) hold 80% of sector revenue.')
print(f'The other {len(df)-n_for_80pct:,} share the remaining 20%.')

1,692 charities (3.8%) hold 80% of sector revenue.
The other 43,151 share the remaining 20%.


In [5]:
# Volunteer leverage by size — sets up the "capacity gap" pillar
df['vol_to_paid'] = df['staff - volunteers'] / df['total full time equivalent staff'].replace(0, np.nan)
by_size = df.groupby('charity size').agg(
    n=('abn_key','count'),
    median_vols=('staff - volunteers','median'),
    median_fte=('total full time equivalent staff','median'),
    median_revenue=('total revenue','median')
).round(0)
by_size

,n,median_vols,median_fte,median_revenue
charity size,,,,
Large,8104,10.00,18.00,"3,418,747.00"
Medium,7367,17.00,2.00,"425,456.00"
Small,29370,10.00,0.00,"17,793.00"


**Read-out:** Small charities run on volunteers — their median FTE is near zero, but volunteer counts are meaningful. That's the population analyst hours can actually help: high volunteer leverage, no in-house analyst capacity.

## 4. Filter to eligible charities

We're not scoring all 45k. Apply defensible exclusions first:

In [6]:
start = len(df)
log = []

df = df[df['conducted activities'] == 'y']
log.append(('Active in reporting year', len(df), start))

df = df[df['registration status'] == 'Registered']
log.append(('Currently registered', len(df), start))

df = df[df['total revenue'].fillna(0) > 0]
log.append(('Has reported revenue', len(df), start))

# Exclude religion-only purposes (advancing religion alone isn't where analytical hours move outcomes)
purpose_cols = ['Preventing_or_relieving_suffering_of_animals','Advancing_Culture','Advancing_Education',
                'Advancing_Health','Promote_or_oppose_a_change_to_law__government_poll_or_prac',
                'Advancing_natual_environment','Promoting_or_protecting_human_rights',
                'Purposes_beneficial_to_ther_general_public_and_other_analogous',
                'Promoting_reconciliation__mutual_respect_and_tolerance','Advancing_Religion',
                'Advancing_social_or_public_welfare','Advancing_security_or_safety_of_Australia_or_Australian_public']
df['purpose_count'] = df[purpose_cols].notna().sum(axis=1)
religion_only = (df['Advancing_Religion'].notna()) & (df['purpose_count'] == 1)
df = df[~religion_only]
log.append(('Not religion-only purpose', len(df), start))

df = df[df['charity size'].isin(['Small','Medium'])]
log.append(('Small or Medium (where capacity gap is real)', len(df), start))

print(f'{'Filter':45s} {'Remaining':>10s}  {'% of start':>10s}')
print('-'*70)
for name, n, s in log:
    print(f'{name:45s} {n:>10,}  {n/s:>9.1%}')
print(f'\nFinal eligible pool: {len(df):,} charities')

Filter                                         Remaining  % of start
----------------------------------------------------------------------
Active in reporting year                          43,439      96.9%
Currently registered                              43,437      96.9%
Has reported revenue                              35,261      78.6%
Not religion-only purpose                         29,861      66.6%
Small or Medium (where capacity gap is real)      22,506      50.2%

Final eligible pool: 22,506 charities


## 5. The 4-pillar scoring model

Each charity gets a 0–100 score across four pillars, then a weighted composite.

| Pillar | What it measures | Weight |
|---|---|---|
| Legitimacy & health | Financial soundness, governance | 25% |
| Impact reach | Beneficiaries, volunteer leverage, geographic spread | 30% |
| Capacity gap | Likelihood that analyst hours fill a real gap | 20% |
| Cause priority | Weighted toward acute need (B-weighting per design) | 25% |

In [7]:
def minmax(s, clip_pct=99):
    """Min-max scale to 0-100, clipping at the 99th percentile to kill outlier dominance."""
    s = s.copy().astype(float)
    cap = np.nanpercentile(s, clip_pct)
    s = s.clip(upper=cap)
    rng = s.max() - s.min()
    if rng == 0: return pd.Series(50, index=s.index)
    return ((s - s.min()) / rng * 100).fillna(0)

In [8]:
# Pillar 1 — Legitimacy & health
df['net_assets_positive'] = (df['net assets/liabilities'].fillna(0) > 0).astype(int) * 100
df['admin_ratio'] = df['employee expenses'].fillna(0) / df['total expenses'].replace(0, np.nan)
df['admin_score'] = 100 - minmax(df['admin_ratio'].fillna(df['admin_ratio'].median()))  # lower admin = better
df['years_operating'] = 2020 - pd.to_datetime(df['Date_Organisation_Established'], errors='coerce').dt.year
df['tenure_score'] = minmax(df['years_operating'].fillna(0).clip(0, 50))

df['pillar_legitimacy'] = (
    0.4 * df['net_assets_positive'] +
    0.3 * df['admin_score'] +
    0.3 * df['tenure_score']
)
df['pillar_legitimacy'].describe().round(1)

count   22,506.00
mean        72.80
std         17.80
min          0.00
25%         67.00
50%         73.60
75%         83.80
max        100.00
Name: pillar_legitimacy, dtype: float64

In [9]:
# Pillar 2 — Impact reach
df['volunteer_score'] = minmax(df['staff - volunteers'].fillna(0))
df['vol_leverage_score'] = minmax(df['vol_to_paid'].replace([np.inf, -np.inf], np.nan).fillna(0))

state_cols = ['Operates_in_ACT','Operates_in_NSW','Operates_in_NT','Operates_in_QLD',
              'Operates_in_SA','Operates_in_TAS','Operates_in_VIC','Operates_in_WA']
df['states_operating'] = df[state_cols].notna().sum(axis=1)
df['geo_score'] = minmax(df['states_operating'])

ben_cols = ['Aboriginal_or_TSI','Adults','Aged_Persons','Children','Communities_Overseas','Early_Childhood',
            'Ethnic_Groups','Families','Females','Financially_Disadvantaged','LGBTIQA+',
            'General_Community_in_Australia','Males','Migrants_Refugees_or_Asylum_Seekers',
            'People_at_risk_of_homelessness','People_with_Chronic_Illness','People_with_Disabilities',
            'Pre_Post_Release_Offenders','Rural_Regional_Remote_Communities','Unemployed_Person',
            'Veterans_or_their_families','Victims_of_crime','Victims_of_Disasters','Youth']
df['beneficiary_count'] = df[ben_cols].notna().sum(axis=1)
df['beneficiary_score'] = minmax(df['beneficiary_count'])

df['pillar_impact'] = (
    0.35 * df['volunteer_score'] +
    0.25 * df['vol_leverage_score'] +
    0.20 * df['geo_score'] +
    0.20 * df['beneficiary_score']
)

In [10]:
# Pillar 3 — Capacity gap (where analyst hours actually help)
# Small > Medium, low FTE = bigger gap, has website = digitally mature enough to use what we build
df['size_gap'] = df['charity size'].map({'Small': 100, 'Medium': 60})
df['fte_inverse'] = 100 - minmax(df['total full time equivalent staff'].fillna(0))
df['has_website'] = df['Charity_Website'].notna().astype(int) * 100

df['pillar_capacity'] = (
    0.45 * df['size_gap'] +
    0.35 * df['fte_inverse'] +
    0.20 * df['has_website']
)

In [11]:
# Pillar 4 — Cause priority (Option B: weighted toward acute need)
# Weights reflect editorial judgment, not arbitrary — defensible in the deck
cause_weights = {
    'People_at_risk_of_homelessness': 3.0,
    'People_with_Disabilities': 3.0,
    'Aboriginal_or_TSI': 3.0,
    'Victims_of_Disasters': 2.5,
    'People_with_Chronic_Illness': 2.5,
    'Financially_Disadvantaged': 2.5,
    'Migrants_Refugees_or_Asylum_Seekers': 2.0,
    'Victims_of_crime': 2.0,
    'Pre_Post_Release_Offenders': 2.0,
    'Veterans_or_their_families': 1.5,
    'Aged_Persons': 1.5,
    'Youth': 1.5,
    'Children': 1.5,
    'Rural_Regional_Remote_Communities': 1.5,
    'Unemployed_Person': 1.5,
}
# Default 1.0 for any beneficiary not listed
for col in ben_cols:
    cause_weights.setdefault(col, 1.0)

df['cause_raw'] = sum(df[col].notna().astype(int) * w for col, w in cause_weights.items())
df['pillar_cause'] = minmax(df['cause_raw'])

In [12]:
# Composite
df['fit_score'] = (
    0.25 * df['pillar_legitimacy'] +
    0.30 * df['pillar_impact'] +
    0.20 * df['pillar_capacity'] +
    0.25 * df['pillar_cause']
).round(1)

df['fit_score'].describe().round(1)

count   22,506.00
mean        44.80
std          9.30
min          9.50
25%         38.50
50%         43.50
75%         50.00
max         96.80
Name: fit_score, dtype: float64

## 6. The top 10

In [13]:
show_cols = ['Charity_Legal_Name','State','charity size','staff - volunteers',
             'total revenue','beneficiary_count','states_operating',
             'pillar_legitimacy','pillar_impact','pillar_capacity','pillar_cause','fit_score']
top10 = df.nlargest(10, 'fit_score')[show_cols].reset_index(drop=True)
top10.round(1)

,Charity_Legal_Name,State,charity size,staff - volunteers,total revenue,beneficiary_count,states_operating,pillar_legitimacy,pillar_impact,pillar_capacity,pillar_cause,fit_score
0,Queensland And Northern New South Wales Lions ...,QLD,Small,1000,156489,23,6,89.90,100.00,96.60,100.00,96.80
1,Australian And New Zealand Burn Association Li...,NSW,Medium,200,330188,22,7,88.40,83.70,78.60,100.00,87.90
2,Nobbys Surf Life Saving Club Inc,NSW,Small,503,95659,20,0,100.00,54.00,100.00,97.30,85.50
3,THE KEEP AUSTRALIA BEAUTIFUL COUNCIL (VIC.) IN...,VIC,Medium,1400,480020,21,1,83.00,84.00,72.60,100.00,85.50
4,Disabled Surfers Association Of Aust,NSW,Small,5000,220277,23,2,85.70,63.00,100.00,100.00,85.30
5,Action for Children And The Aged (ACATA) Austr...,QLD,Small,500,2966,20,1,70.00,83.00,98.30,89.00,84.30
6,Knit4Charities Inc,QLD,Small,2000,14842,21,2,79.60,63.00,100.00,100.00,83.80
7,Art Of Living Foundation Ltd,NSW,Medium,750,639960,20,7,79.60,74.00,82.00,100.00,83.50
8,Warriewood Beach Surf Life Saving Club Inc,NSW,Medium,800,253770,21,0,100.00,55.00,82.00,100.00,82.90
9,Churches Of Christ Community Care,VIC,Medium,250,808947,19,0,97.60,66.40,78.60,89.00,82.30


## 7. Sensitivity check

If the pillar weights are arbitrary, the shortlist is fragile. Re-score with equal weights and see how much the top 10 shifts.

In [14]:
df['fit_equal'] = (df['pillar_legitimacy'] + df['pillar_impact'] + df['pillar_capacity'] + df['pillar_cause']) / 4
top10_equal = set(df.nlargest(10, 'fit_equal')['abn_key'])
top10_design = set(df.nlargest(10, 'fit_score')['abn_key'])
overlap = len(top10_equal & top10_design)
print(f'Overlap between designed weighting and equal weighting: {overlap}/10')
print(f'Robust core: {overlap} charities appear regardless of weighting choice.')

Overlap between designed weighting and equal weighting: 8/10
Robust core: 8 charities appear regardless of weighting choice.


## 8. Export for Tableau

In [15]:
# Full scored dataset for the dashboard (top-of-list filterable in Tableau)
export_cols = [
    'abn_key','Charity_Legal_Name','State','Town_City','Charity_Website','charity size',
    'staff - volunteers','total full time equivalent staff','total revenue','net assets/liabilities',
    'beneficiary_count','states_operating','years_operating',
    'pillar_legitimacy','pillar_impact','pillar_capacity','pillar_cause','fit_score'
] + [c for c in ben_cols if c in df.columns] + [c for c in purpose_cols if c in df.columns]

out = df[export_cols].sort_values('fit_score', ascending=False).reset_index(drop=True)
out['rank'] = out.index + 1
out['is_top_10'] = (out['rank'] <= 10).astype(int)
out.to_csv('charity_shortlist.csv', index=False)
print(f'Exported {len(out):,} scored charities to charity_shortlist.csv')
print(f'Top 10 flagged via is_top_10 column.')

Exported 22,506 scored charities to charity_shortlist.csv
Top 10 flagged via is_top_10 column.


## 9. What we'd actually deliver in 2 hrs/week

The shortlist is the *who*. For the deck, each of the top 10 gets a one-line *what* — the analytical work we'd deliver. Examples by charity profile:

- **Disability services** → outcome dashboard (NDIS participant journey, service gap heatmap)
- **Homelessness** → demand forecasting from referral data, bed-night optimization
- **Aboriginal/TSI** → community-program impact measurement, grant-reporting automation
- **Disaster response** → preparedness mapping, donor-retention cohort analysis
- **Environmental** → volunteer-hour productivity, geospatial site prioritization

These aren't generic — they map to what the AIS data actually shows each charity does.